# Eval error distributions

Absolute `|final_xy_err_m|` and `|final_yaw_err_rad|` histograms from an eval `results.csv`.
Rows: log-binned, linear-binned, and one bar per episode. Each bar is coloured by mean IoU (or that episode's IoU).

In [ ]:
from __future__ import annotations

import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# --- paths / knobs (edit these) ---
RESULTS_CSV = Path(
    "/home/ubuntu/workspace/camelo-ebim/outputs/eval/max_rad_0_005_max_xy_0_02/results.csv"
)

# Number of log-spaced bins for each absolute-error histogram.
# Change these to coarsen / refine the plots.
N_BINS_XY = 8
N_BINS_YAW = 8

# Optional hard edges (None => data-driven with a small pad).
XY_ERR_RANGE_M: tuple[float, float] | None = None
YAW_ERR_RANGE_RAD: tuple[float, float] | None = None

In [ ]:
df = pd.read_csv(RESULTS_CSV)
n_all = len(df)
# Timeouts / failures fill `error` and leave `loop_approach` empty.
if "error" in df.columns:
    df = df[df["error"].isna() & df["loop_approach"].notna()].copy()
else:
    df = df[df["loop_approach"].notna()].copy()
print(f"kept {len(df)}/{n_all} episodes (dropped timeouts/errors) from {RESULTS_CSV}")

approach = df["loop_approach"].map(ast.literal_eval)

df["final_xy_err_m"] = approach.map(lambda d: float(d["final_xy_err_m"]))
df["final_yaw_err_rad"] = approach.map(lambda d: float(d["final_yaw_err_rad"]))
df["abs_xy_err_m"] = df["final_xy_err_m"].abs()
df["abs_yaw_err_rad"] = df["final_yaw_err_rad"].abs()

display(
    df[["episode", "iou", "abs_xy_err_m", "abs_yaw_err_rad", "orientation_case"]].describe(
        include="all"
    )
)
df[["episode", "iou", "abs_xy_err_m", "abs_yaw_err_rad", "orientation_case"]]

In [ ]:
def bin_edges(
    values: np.ndarray,
    n_bins: int,
    explicit_range: tuple[float, float] | None,
    *,
    scale: str,
) -> np.ndarray:
    """One-sided positive bin edges covering the absolute errors."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals) & (vals >= 0)]
    if vals.size == 0:
        raise ValueError("no non-negative absolute errors to bin")

    if explicit_range is None:
        lo, hi = float(vals.min()), float(vals.max())
        if scale == "log":
            # Log scale needs strictly positive edges; pad so min/max sit inside.
            positive = vals[vals > 0]
            if positive.size == 0:
                raise ValueError("no positive absolute errors for log bins")
            lo, hi = float(positive.min()), float(positive.max())
            lo = max(lo / 1.2, np.finfo(float).tiny)
            hi = hi * 1.2
        else:
            # Linear: start at 0 (one-sided absolute error) and pad the top.
            lo = 0.0
            hi = hi * 1.05 if hi > 0 else 1.0
    else:
        lo, hi = explicit_range
        if hi <= lo:
            raise ValueError(f"need lo < hi, got {explicit_range}")
        if scale == "log" and lo <= 0:
            raise ValueError(f"log scale needs lo > 0, got {explicit_range}")

    if scale == "log":
        return np.geomspace(lo, hi, n_bins + 1)
    if scale == "linear":
        return np.linspace(lo, hi, n_bins + 1)
    raise ValueError(f"scale must be 'log' or 'linear', got {scale!r}")


def _iou_norm_cmap(ious: np.ndarray, cmap_name: str):
    finite = ious[np.isfinite(ious)]
    vmin = float(finite.min()) if finite.size else 0.0
    vmax = float(finite.max()) if finite.size else 1.0
    if vmin == vmax:
        vmax = vmin + 1e-9
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap(cmap_name)
    return norm, cmap


def iou_coloured_hist(
    ax,
    errors: np.ndarray,
    ious: np.ndarray,
    *,
    n_bins: int,
    err_range: tuple[float, float] | None,
    xlabel: str,
    title: str,
    scale: str = "log",
    cmap_name: str = "viridis",
) -> ScalarMappable:
    errors = np.asarray(errors, dtype=float)
    ious = np.asarray(ious, dtype=float)
    edges = bin_edges(errors, n_bins, err_range, scale=scale)
    counts, _ = np.histogram(errors, bins=edges)

    # Mean IoU per bin (NaN when empty).
    bin_idx = np.digitize(errors, edges) - 1
    mean_iou = np.full(n_bins, np.nan)
    for i in range(n_bins):
        mask = bin_idx == i
        if mask.any():
            mean_iou[i] = float(np.nanmean(ious[mask]))

    norm, cmap = _iou_norm_cmap(mean_iou[np.isfinite(mean_iou)], cmap_name)
    widths = np.diff(edges)
    colors = [
        cmap(norm(mi)) if np.isfinite(mi) else (0.9, 0.9, 0.9, 1.0) for mi in mean_iou
    ]

    ax.bar(
        edges[:-1],
        counts,
        width=widths,
        align="edge",
        color=colors,
        edgecolor="black",
        linewidth=0.6,
    )
    ax.set_xscale(scale)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("count")
    ax.set_title(title)
    ax.set_ylim(bottom=0)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    return sm


def iou_coloured_per_episode(
    ax,
    errors: np.ndarray,
    ious: np.ndarray,
    *,
    xlabel: str,
    title: str,
    cmap_name: str = "viridis",
    width_frac: float = 0.55,
) -> ScalarMappable:
    """One bar per episode at its absolute-error x, coloured by that episode's IoU."""
    errors = np.asarray(errors, dtype=float)
    ious = np.asarray(ious, dtype=float)
    order = np.argsort(errors)
    errors = errors[order]
    ious = ious[order]

    # Width from nearest-neighbour gap so equal-valued runs still get a visible bar.
    if errors.size == 1:
        widths = np.array([max(errors[0] * 0.05, 1e-6)])
    else:
        gaps = np.diff(errors)
        pos = gaps[gaps > 0]
        fallback = float(pos.min()) if pos.size else max(float(errors.max()) * 0.05, 1e-6)
        left = np.empty(errors.size)
        right = np.empty(errors.size)
        left[0] = fallback
        right[-1] = fallback
        left[1:] = np.where(gaps > 0, gaps, fallback)
        right[:-1] = left[1:]
        widths = width_frac * np.minimum(left, right)

    norm, cmap = _iou_norm_cmap(ious, cmap_name)
    colors = [cmap(norm(i)) for i in ious]
    ax.bar(
        errors,
        np.ones_like(errors),
        width=widths,
        align="center",
        color=colors,
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_xscale("linear")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("count")
    ax.set_title(title)
    ax.set_ylim(0, 1.15)
    ax.set_yticks([0, 1])

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    return sm


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 11.5), constrained_layout=True)

hist_specs = [
    (axes[0, 0], "abs_xy_err_m", N_BINS_XY, XY_ERR_RANGE_M, "|final_xy_err| (m)", "XY error (log)", "log"),
    (axes[0, 1], "abs_yaw_err_rad", N_BINS_YAW, YAW_ERR_RANGE_RAD, "|final_yaw_err| (rad)", "yaw error (log)", "log"),
    (axes[1, 0], "abs_xy_err_m", N_BINS_XY, XY_ERR_RANGE_M, "|final_xy_err| (m)", "XY error (linear)", "linear"),
    (axes[1, 1], "abs_yaw_err_rad", N_BINS_YAW, YAW_ERR_RANGE_RAD, "|final_yaw_err| (rad)", "yaw error (linear)", "linear"),
]

for ax, col, n_bins, err_range, xlabel, title, scale in hist_specs:
    sm = iou_coloured_hist(
        ax,
        df[col].to_numpy(),
        df["iou"].to_numpy(),
        n_bins=n_bins,
        err_range=err_range,
        xlabel=xlabel,
        title=title,
        scale=scale,
    )
    fig.colorbar(sm, ax=ax, label="mean IoU in bin")

for ax, col, xlabel, title in [
    (axes[2, 0], "abs_xy_err_m", "|final_xy_err| (m)", "XY error (one bar per episode)"),
    (axes[2, 1], "abs_yaw_err_rad", "|final_yaw_err| (rad)", "yaw error (one bar per episode)"),
]:
    sm = iou_coloured_per_episode(
        ax,
        df[col].to_numpy(),
        df["iou"].to_numpy(),
        xlabel=xlabel,
        title=title,
    )
    fig.colorbar(sm, ax=ax, label="episode IoU")

fig.suptitle(RESULTS_CSV.parent.name, fontsize=11)
plt.show()
